In [2]:
from math import comb
from typing import List

import numpy as np



In [3]:
def probability_exact_X_maxY(
        n: List[int],  # n[k] = number of balls of type k
        p: List[List[float]],  # p[k][j] = prob bucket j for type k
        X: int,  # target count in bucket 0
        Y: int  # target MAX in any bucket 1 ... U-2
) -> float:
    K = len(n)
    U = len(p[0])

    total_prob = 0.0

    def recurse(k: int, current_counts: np.ndarray, prob_so_far: float):
        nonlocal total_prob

        if k == K:
            if current_counts[0] == X:
                max_other = np.max(current_counts[1:-1])
                if max_other == Y:
                    total_prob += prob_so_far
            return

        nk = n[k]
        pk = p[k]

        def gen_counts(pos: int, rem: int, current_this: List[int]):
            if pos == U:
                if rem == 0:
                    logp = 0.0
                    multinom = 1
                    for j in range(U):
                        multinom *= comb(rem if j == 0 else current_this[j - 1], current_this[j]) if j > 0 else 1
                        logp += current_this[j] * np.log(pk[j] + 1e-300)
                    multinom_coeff = np.math.factorial(nk) / np.prod([np.math.factorial(c) for c in current_this])
                    this_prob = multinom_coeff * np.exp(logp)

                    new_counts = current_counts + np.array(current_this)
                    recurse(k + 1, new_counts, prob_so_far * this_prob)
                return

            for c in range(rem + 1):
                current_this[pos] = c
                gen_counts(pos + 1, rem - c, current_this)

        this_counts = [0] * U
        gen_counts(0, nk, this_counts)

    initial_counts = np.zeros(U, dtype=int)
    recurse(0, initial_counts, 1.0)

    return total_prob